# Full Pipeline Dummy (Single Notebook)
Notebook ini menunjukkan alur lengkap NR-IQA dalam satu file: load data -> FFT kecil -> CLIP dummy -> fusion -> prediksi -> evaluasi dummy.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
manifest = pd.read_csv(ROOT / 'data' / 'genimage_manifest.csv')
labels = pd.read_csv(ROOT / 'data' / 'labels_quality.csv')
manifest.head()

## FFT (sedikit tapi nyata)
Bagian ini pakai `np.fft.fft2` + `fftshift`, lalu ambil energi low/mid/high frekuensi dari pseudo-image deterministic berdasarkan `image_id`.

In [ ]:
import hashlib

def dummy_image(image_id, size=64):
    seed = int.from_bytes(hashlib.sha256(image_id.encode('utf-8')).digest()[:8], 'little')
    rng = np.random.default_rng(seed)
    img = rng.normal(127, 40, (size, size)).astype(np.float32)
    return np.clip(img, 0, 255)

def fft_features(img):
    f = np.fft.fft2(img)
    fs = np.fft.fftshift(f)
    mag = np.abs(fs)

    h, w = img.shape
    y, x = np.ogrid[:h, :w]
    cy, cx = h // 2, w // 2
    r = np.sqrt((y - cy) ** 2 + (x - cx) ** 2)
    rn = r / (r.max() + 1e-8)

    return {
        'fft_mag_mean': float(mag.mean()),
        'fft_mag_std': float(mag.std()),
        'fft_low_energy': float(mag[rn <= 0.2].mean()),
        'fft_mid_energy': float(mag[(rn > 0.2) & (rn <= 0.5)].mean()),
        'fft_high_energy': float(mag[rn > 0.5].mean()),
    }

In [ ]:
fft_rows = []
for image_id in manifest['image_id']:
    fft_rows.append({'image_id': image_id, **fft_features(dummy_image(image_id))})
fft_df = pd.DataFrame(fft_rows)
fft_df.head()

## CLIP Dummy + Fusion + Dummy Regressor

In [ ]:
def clip_dummy(image_id):
    seed = int.from_bytes(hashlib.md5(image_id.encode('utf-8')).digest()[:8], 'little')
    rng = np.random.default_rng(seed)
    v = rng.random(3)
    return {'clip_feat_001': float(v[0]), 'clip_feat_002': float(v[1]), 'clip_feat_003': float(v[2])}

clip_df = pd.DataFrame([{'image_id': i, **clip_dummy(i)} for i in manifest['image_id']])
fused = fft_df.merge(clip_df, on='image_id', how='inner')
fused['pred_score'] = 0.35*fused['fft_low_energy'] + 0.25*fused['fft_mid_energy'] + 0.15*fused['fft_high_energy'] + 0.25*fused['clip_feat_001']
fused['pred_score'] = (fused['pred_score'] - fused['pred_score'].min()) / (fused['pred_score'].max() - fused['pred_score'].min() + 1e-8)
fused['pred_label_optional'] = np.where(fused['pred_score'] >= 0.5, 'real_like', 'ai_like')
fused[['image_id','pred_score','pred_label_optional']].head()

In [ ]:
eval_df = fused[['image_id','pred_score']].merge(labels, on='image_id', how='inner')
dummy_srocc = eval_df['pred_score'].corr(eval_df['quality_score'], method='spearman')
dummy_mae = (eval_df['pred_score'] - eval_df['quality_score']).abs().mean()
pd.DataFrame([{'metric':'dummy_srocc','value':float(dummy_srocc) if dummy_srocc==dummy_srocc else 0.0}, {'metric':'dummy_mae','value':float(dummy_mae)}])

## Opsional Gemini
Setelah prediksi, pilih subset `high_error` atau `high_uncertainty` untuk evaluasi manual di Gemini, lalu simpan hasil ke `artifacts/gemini_eval_dummy.jsonl`.